# 14 · Powerlifting：第四轮特征工程——把「这个人有多强」放进模型（Colab GPU，五折）

**为什么还能做**：前几轮的特征（记录精度、比赛背景、运动员历史计数）都是**不看成绩**的描述；它们已经接近饱和，剩下的分组偏差多在 2 kg 以内。但效果最大的一步——同一运动员平滑——用的是**同一个人在训练集里的真实成绩**，而且目前只在模型之后做线性修正（S3 −0.14），模型本身看不到。本轮把这类**个人层面、基于成绩**的信息直接做成特征，让树模型非线性地使用（例如：间隔越短、同一装备、记录越多，越可信）。

为了不泄漏，所有基于成绩的特征都用**嵌套折外**计算：外层每一折里，训练部分再分 5 个内层折，每行的特征只用其他内层折的成绩；外层验证行和测试行用整个训练部分。验证行的成绩从不参与它自己的特征。

|块|内容|
|---|---|
|`AS` 运动员实力|近似同一个人（性别 + 国家 + 州 [+ 联盟] [+ 装备] + 出生年 ±1）其他记录的相对力量（DOTS 分）：个数、均值、最高值；时间衰减均值（离这次比赛越近权重越大）、与最近一次的年份差、平均年份差；体重变化；同装备记录比例；双胞胎（16 列 / 核心 9 列相同）的总成绩均值与个数|
|`MS` 比赛实力|同一场比赛（联盟 × 年份 × 比赛国 × 比赛州）以及同一组别（再 × 性别 × 装备）**其他选手**的相对力量：个数、均值、最高值|
|`ID`（视检查结果）|如果 `id` 的顺序里藏着比赛结构（第 6 节检查），加入 `id` 作为特征|

**另外两项**
- **`id` 结构检查**：按 `id` 排序后，相邻两行来自同一场比赛的比例是否明显高于随机。若是，说明 `id` 保留了原始数据的顺序，就把它当特征一起测试。
- **目标变量工程**：目前最强的单模型是「DOTS 相对力量」目标的 CatBoost。本轮再试两种力量公式作为目标：异速生长（总成绩 ÷ 体重^(2/3)）和 IPF GL 系数（按性别 × 是否装备）。看它们单独的分数，以及加入融合后的增益。

**协议**：种子 42 的五折（与 07–13 相同），内层 10% 早停；模型 LightGBM、XGBoost、cb_rel（三个融合）。基线 `R3` = 11 的 ALL3 分模型特征。达标的块（三模型融合平均 Δ ≤ −0.05 且 ≥ 4/5 折）自动合成 `ALL4`。最后在 `R3` 与 `ALL4` 上都加双胞胎 + S3 平滑，看**扣除平滑之后的净收益**（新特征与平滑有重叠）。导出两个提交文件做配对比较。

**操作**：Colab 选 **T4 GPU**，从上到下运行；约 1–1.5 小时，可续跑。

## 1. 环境与配置（与 11 相同，实验名不同）

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import gc
import hashlib
import importlib
import importlib.metadata
import inspect
import json
import os
import shutil
import subprocess
import sys
import time
import warnings

try:
    from google.colab import drive
except ModuleNotFoundError:
    drive = None
if drive is not None:
    drive.mount('/content/drive')

PROJECT_DIR = Path(os.environ.get(
    'PL_PROJECT_DIR',
    '/content/drive/MyDrive/Colab Notebooks/performances-en-powerlifting',
)).expanduser().resolve()
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(f'找不到项目目录：{PROJECT_DIR}；请修改 PROJECT_DIR')

CATBOOST_VERSION = '1.2.10'
to_install = []
try:
    installed_catboost = importlib.metadata.version('catboost')
except importlib.metadata.PackageNotFoundError:
    installed_catboost = None
if installed_catboost != CATBOOST_VERSION:
    if 'catboost' in sys.modules:
        raise RuntimeError('当前运行时已经导入另一版 CatBoost；请重启运行时后从头运行')
    to_install.append(f'catboost=={CATBOOST_VERSION}')
try:
    importlib.metadata.version('pytabkit')
except importlib.metadata.PackageNotFoundError:
    to_install.append('pytabkit')
if to_install:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *to_install])
    importlib.invalidate_caches()

import catboost
import lightgbm as lgb
import numpy as np
import pandas as pd
import torch
import xgboost as xgb
from IPython.display import display
from scipy.optimize import nnls
from sklearn.model_selection import KFold, train_test_split

try:
    from pytabkit import TabM_D_Regressor
    PYTABKIT_ERROR = None
except Exception as exc:                       # 安装或导入失败时其余模型照常运行
    TabM_D_Regressor = None
    PYTABKIT_ERROR = repr(exc)

warnings.filterwarnings('ignore', category=UserWarning)
if catboost.__version__ != CATBOOST_VERSION:
    raise RuntimeError(f'需要 CatBoost {CATBOOST_VERSION}；实际为 {catboost.__version__}')
if shutil.which('nvidia-smi') is None or not torch.cuda.is_available():
    raise RuntimeError('没有可用的 GPU：请在 Colab 选择 T4 GPU 后从头运行')
GPU_NAME = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                          text=True, capture_output=True, check=False).stdout.strip().splitlines()[0]

EXPERIMENT_ID = 'feature_round4_v1'
OUTPUT_DIR = PROJECT_DIR / 'artifacts' / EXPERIMENT_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42                                  # 07–10 的五折划分种子（第一个种子）
SEEDS = [42, 7, 2024]                      # 每个种子：不同的五折划分、内层早停划分和模型随机性
N_FOLDS = 5
INNER_STOP_FRACTION = 0.10
MAX_ROUNDS = 20000
FINAL_MODELS = ['lgb', 'xgb', 'lgb_rel', 'catboost', 'cb_rel', 'tabm']   # 每折内的运行顺序：快的先跑
FULL_REFIT = True                          # 树模型在全部训练行上重训
REFIT_FACTOR = 1.2                         # 全量重训轮数 = CV 平均最佳轮数 × 1.2（数据多约 39%）
RUN_CLEANING_TEST = True                   # 第 9 节：数据清洗实验（只作报告证据）
MODELS = ['lgb', 'xgb', 'catboost']        # 09 / 10 特征单元使用的三个模型族名
RUN_ONLY = None                            # 09 / 10 特征单元需要的变量，这里不用
KEEP_DELTA = -0.05
EXPECTED_SHA256 = {'train.csv': '62319814116506fb95a51f1185dbacdb170ab055642ee4c347a30584093d0856', 'test.csv': '563c6fa29f9e535740ef1f09b25954f00fbe242c81c0e5a7343dc18aa962ba46'}
VERSIONS = {'catboost': catboost.__version__, 'xgboost': xgb.__version__, 'lightgbm': lgb.__version__,
            'torch': torch.__version__, 'pytabkit': None if PYTABKIT_ERROR else importlib.metadata.version('pytabkit'),
            'numpy': np.__version__, 'pandas': pd.__version__, 'python': sys.version.split()[0]}
print('PROJECT_DIR :', PROJECT_DIR)
print('OUTPUT_DIR  :', OUTPUT_DIR)
print('GPU         :', GPU_NAME)
print('版本         :', VERSIONS)
if PYTABKIT_ERROR:
    print('pytabkit 不可用，TabM 会在预检中跳过：', PYTABKIT_ERROR)

SCREEN_MODELS = ['lgb', 'xgb', 'cb_rel']        # 筛选用的三个模型（cb_rel 是最强的单模型）
TARGET_MODELS = ['cb_allo', 'cb_gl']            # 目标变量工程：两种力量公式
INNER_FEATURE_FOLDS = 5                         # 基于成绩的特征：外层训练部分内的嵌套折数

Mounted at /content/drive
PROJECT_DIR : /content/drive/MyDrive/Colab Notebooks/performances-en-powerlifting
OUTPUT_DIR  : /content/drive/MyDrive/Colab Notebooks/performances-en-powerlifting/artifacts/feature_round4_v1
GPU         : Tesla T4, 15360 MiB
版本         : {'catboost': '1.2.10', 'xgboost': '3.4.1', 'lightgbm': '4.6.0', 'torch': '2.11.0+cu130', 'pytabkit': '1.7.3', 'numpy': '2.1.3', 'pandas': '2.2.3', 'python': '3.13.15'}


## 2. 读取并核对数据（与 07 相同）

In [2]:
def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def rmse(a, b):
    a, b = np.asarray(a, dtype='float64'), np.asarray(b, dtype='float64')
    return float(np.sqrt(np.mean((a - b) ** 2)))


for name, expected in EXPECTED_SHA256.items():
    require(sha256_file(PROJECT_DIR / name) == expected, f'{name} 与本地版本不同；请检查 Drive 上的数据')

TARGET = 'TotalKg'
train = pd.read_csv(PROJECT_DIR / 'train.csv')
test = pd.read_csv(PROJECT_DIR / 'test.csv')
RAW_NUM = ['Age', 'BodyweightKg', 'MeetYear']
RAW_FEATURES = [c for c in test.columns if c != 'id']
CAT = [c for c in RAW_FEATURES if c not in RAW_NUM]          # 不依赖 object / str dtype（pandas 2 与 3 都可用）
require(train.shape == (105000, 18) and test.shape == (45000, 17), '数据形状不符')
require(len(CAT) == 13 and all(pd.api.types.is_numeric_dtype(train[c]) for c in RAW_NUM)
        and not any(pd.api.types.is_numeric_dtype(train[c]) for c in CAT), '列分组不符')
require(not train[RAW_FEATURES].isna().any().any() and not test[RAW_FEATURES].isna().any().any(), '出现空值')
y = train[TARGET].to_numpy(dtype='float64')
n_train = len(train)

FOLD_IDS = np.zeros(n_train, dtype=int)
for k, (_, valid) in enumerate(KFold(N_FOLDS, shuffle=True, random_state=SEED).split(np.arange(n_train)), start=1):
    FOLD_IDS[valid] = k
print(f'train {train.shape}  test {test.shape}；每折验证行数：{np.bincount(FOLD_IDS)[1:].tolist()}')

train (105000, 18)  test (45000, 17)；每折验证行数：[21000, 21000, 21000, 21000, 21000]


## 3. 第一轮特征（与 07 相同）

In [3]:
both = pd.concat([train[RAW_FEATURES], test[RAW_FEATURES]], ignore_index=True)

# F0：v2 的体重级别特征
weight_class = pd.to_numeric(both['WeightClassKg'].str.replace('+', '', regex=False), errors='coerce')
F0_EXTRA = pd.DataFrame({
    'WeightClass_num': weight_class,
    'Is_SHW': both['WeightClassKg'].str.endswith('+').astype('float64'),
    'BW_minus_Class': both['BodyweightKg'] - weight_class,
})

# P：记录精度（树按阈值切数值，学不到「是不是 .5」「是不是整数」）
age2 = (both['Age'] * 2).round()
require(np.allclose(both['Age'] * 2, age2), 'Age 不是 0.5 的整数倍')
bw = both['BodyweightKg']
is_int = np.isclose(bw, bw.round())
is_1dec = np.isclose(bw * 10, (bw * 10).round())
is_half = is_1dec & ~is_int & np.isclose((bw * 10).round() % 10, 5)
pounds = bw / 0.45359237
P = pd.DataFrame({
    'age_half': (age2 % 2).astype('float64'),
    'bw_precision': np.select([is_int, is_half, is_1dec], [0.0, 1.0, 2.0], default=3.0),
    'bw_lb_int': (np.abs(pounds - pounds.round()) < 0.015).astype('float64'),
})

# T：跨地参赛（居住地未知时为 NaN）
abroad = (both['Country'] != both['MeetCountry']).astype('float64').where(both['Country'] != 'Unknown')
state_known = ((both['State'] != 'Unknown') & (both['MeetState'] != 'Unknown')
               & (both['Country'] == both['MeetCountry']))
other_state = (both['State'] != both['MeetState']).astype('float64').where(state_known)
T = pd.DataFrame({'abroad': abroad, 'other_state': other_state})

# M：比赛背景；「比赛」≈ 联盟 × 年份 × 比赛国 × 比赛州（train+test 无标签统计）
meet = (both['Federation'] + '|' + both['MeetYear'].astype(str) + '|'
        + both['MeetCountry'] + '|' + both['MeetState'])


def by_meet(values):
    return values.groupby(meet).transform('mean')


M = pd.DataFrame({
    'meet_n': meet.map(meet.value_counts()).astype('float64'),
    'meet_sexeq_n': both.groupby([meet, both['Sex'], both['Equipment']])['Age'].transform('size').astype('float64'),
    'fed_year_n': both.groupby(['Federation', 'MeetYear'])['Age'].transform('size').astype('float64'),
    'meet_share_other_state': by_meet(other_state),
    'meet_share_abroad': by_meet(abroad),
    'meet_share_age_half': by_meet(P['age_half']),
    'meet_share_equipped': by_meet((both['Equipment'] != 'Raw').astype('float64')),
    'meet_share_male': by_meet((both['Sex'] == 'M').astype('float64')),
    'meet_mean_age': by_meet(both['Age']),
})

BLOCKS = {'P': P, 'T': T, 'M': M}
VARIANTS = {'F0': [], 'P': ['P'], 'T': ['T'], 'M': ['M'], 'PT': ['P', 'T'], 'ALL': ['P', 'T', 'M']}
for name, frame in {'F0': F0_EXTRA, **BLOCKS}.items():
    require(len(frame) == len(both) and frame.index.equals(both.index), f'{name} 行数或索引不符')
    require(not np.isinf(frame.to_numpy(dtype='float64')).any(), f'{name} 含无穷大')

_FRAMES = {}


def frames_for(variant, model):
    """(train, test) 特征表。CatBoost 的类别为字符串；LGB / XGB 为 category（词表来自 train+test，无标签）。"""
    kind = 'str' if model == 'catboost' else 'category'
    if (variant, kind) not in _FRAMES:
        frame = pd.concat([both[RAW_NUM], F0_EXTRA] + [BLOCKS[b] for b in VARIANTS[variant]] + [both[CAT]], axis=1)
        require(frame.columns.is_unique, f'{variant} 列名重复')
        frame[CAT] = frame[CAT].astype(str).astype(object) if kind == 'str' else frame[CAT].astype('category')
        _FRAMES[(variant, kind)] = (frame.iloc[:n_train].reset_index(drop=True),
                                    frame.iloc[n_train:].reset_index(drop=True))
    return _FRAMES[(variant, kind)]


print({v: frames_for(v, 'lgb')[0].shape[1] for v in VARIANTS})
summary_new = pd.concat([P, T, M], axis=1)
display(pd.DataFrame({'train_mean': summary_new.iloc[:n_train].mean(), 'test_mean': summary_new.iloc[n_train:].mean(),
                      'nan_share': summary_new.isna().mean()}).round(4))

{'F0': 19, 'P': 22, 'T': 21, 'M': 28, 'PT': 24, 'ALL': 33}


,train_mean,test_mean,nan_share
age_half,0.4817,0.4782,0.0000
bw_precision,1.9668,1.9609,0.0000
bw_lb_int,0.0500,0.0503,0.0000
abroad,0.0824,0.0822,0.0184
other_state,0.2670,0.2724,0.6491
meet_n,63.6623,63.9177,0.0000
meet_sexeq_n,33.0578,33.1368,0.0000
fed_year_n,653.6460,656.8197,0.0000
meet_share_other_state,0.2661,0.2673,0.6219
meet_share_abroad,0.0833,0.0837,0.0004


## 4. 第二轮特征（与 09 相同）

In [4]:
_FRAMES.clear()                     # 07 的变体表不再需要，释放内存
CORE9 = ['Age', 'BodyweightKg', 'MeetYear', 'Sex', 'Federation', 'MeetCountry', 'MeetState', 'Country', 'State']
sex, equipment = both['Sex'], both['Equipment']
state = both['State'].where(both['State'] != 'Unknown')
country = both['Country'].where(both['Country'] != 'Unknown')
national = both['Federation'] + '|' + both['MeetYear'].astype(str) + '|' + both['MeetCountry']


def counts(key):
    return key.map(key.value_counts()).astype('float64')


def n_distinct(values, key):
    return values.groupby(key).transform('nunique').astype('float64')


# M2：比赛背景加强（meet 与 by_meet 来自 07 的特征单元）
M2 = pd.DataFrame({
    'meet_n_states': n_distinct(state, meet),
    'meet_n_countries': n_distinct(country, meet),
    'meet_n_wclass': n_distinct(both['WeightClassKg'], meet),
    'meet_share_tested': by_meet((both['Tested'] == 'Yes').astype('float64')),
    'meet_share_raw': by_meet((equipment == 'Raw').astype('float64')),
    'meet_share_multi': by_meet((equipment == 'Multi-ply').astype('float64')),
    'meet_share_bw2dec': by_meet((P['bw_precision'] == 3).astype('float64')),
    'meet_share_junior': by_meet((both['Age'] < 24).astype('float64')),
    'meet_share_master': by_meet((both['Age'] >= 40).astype('float64')),
    'meet_age_std': both['Age'].groupby(meet).transform('std'),
    'meet_sex_bw_mean': both['BodyweightKg'].groupby([meet, sex]).transform('mean'),
    'nat_n': counts(national),
    'nat_n_meetstates': n_distinct(both['MeetState'], national),
    'nat_n_states': n_distinct(state, national),
})

# D：组别背景
division = meet + '|' + sex + '|' + equipment + '|' + both['WeightClassKg']
age_division = meet + '|' + sex + '|' + equipment + '|' + both['AgeClass']
exact_key = pd.Series(pd.util.hash_pandas_object(both[RAW_FEATURES], index=False).to_numpy())
core_key = pd.Series(pd.util.hash_pandas_object(both[CORE9], index=False).to_numpy())
D = pd.DataFrame({
    'div_n': counts(division),
    'div_bw_rank': both['BodyweightKg'].groupby(division).rank(pct=True),
    'div_age_rank': both['Age'].groupby(division).rank(pct=True),
    'agediv_n': counts(age_division),
    'dup_exact_n': counts(exact_key) - 1,
    'dup_core_n': counts(core_key) - 1,
})

# H：近似同一运动员的历史（出生年 = floor(MeetYear − Age)：x.5 年龄时恰为出生年，整数年龄可能差 1）
valid = (both['Country'] != 'Unknown').to_numpy()
birth = np.floor(both['MeetYear'] - both['Age'] + 1e-9).astype(int).astype(str)
athlete = sex + '|' + both['Country'] + '|' + both['State'] + '|' + both['Federation'] + '|' + birth
athlete0 = sex + '|' + both['Country'] + '|' + both['State'] + '|' + birth
year = both['MeetYear'].astype('float64')
H = pd.DataFrame({
    'ath_n': counts(athlete),
    'ath_n_years': n_distinct(year, athlete),
    'ath_years_since_first': year - year.groupby(athlete).transform('min'),
    'ath_years_to_last': year.groupby(athlete).transform('max') - year,
    'ath_n_equipment': n_distinct(equipment, athlete),
    'ath_bw_delta': both['BodyweightKg'] - both['BodyweightKg'].groupby(athlete).transform('mean'),
    'ath0_n': counts(athlete0),
    'ath0_n_feds': n_distinct(both['Federation'], athlete0),
})
H.loc[~valid] = np.nan


# S：DOTS 系数（TotalKg = DOTS 分 × dots_scale）
def dots_scale(sex_values, bw_values):
    s = np.asarray(sex_values).astype(str)
    bw = np.asarray(bw_values, dtype='float64')
    m, f = np.clip(bw, 40, 210), np.clip(bw, 40, 150)
    poly_m = -1.0930e-6 * m**4 + 7.391293e-4 * m**3 - 0.1918759221 * m**2 + 24.0900756 * m - 307.75076
    poly_f = -1.0706e-6 * f**4 + 5.158568e-4 * f**3 - 0.1126655495 * f**2 + 13.6175032 * f - 57.96288
    return np.where(s == 'M', poly_m, np.where(s == 'F', poly_f, 0.5 * (poly_m + poly_f))) / 500.0


S = pd.DataFrame({'dots_scale': dots_scale(sex, both['BodyweightKg'])})

# C：只给 CatBoost 的高基数 ID（有序目标统计编码）
C = pd.DataFrame({'meet_id': meet, 'athlete_id': (athlete + '|' + equipment).where(valid, 'unknown')})

BLOCKS2 = {'M2': M2, 'D': D, 'H': H, 'S': S}
CB_ONLY = {'C': C}
for name, frame in {**BLOCKS2, **CB_ONLY}.items():
    require(len(frame) == len(both) and frame.index.equals(both.index), f'{name} 行数或索引不符')
for name, frame in BLOCKS2.items():
    require(not np.isinf(frame.to_numpy(dtype='float64')).any(), f'{name} 含无穷大')
require(S['dots_scale'].min() > 0.3, 'DOTS 系数异常')

R1_BLOCKS = ['P', 'T', 'M']
VARIANT_BLOCKS = {'R1': [], 'M2': ['M2'], 'D': ['D'], 'H': ['H'], 'S': ['S'], 'C': ['C']}
MODELS_FOR = {'C': ['catboost']}      # C 只改变 CatBoost；融合时 LightGBM / XGBoost 用 R1
SCREEN = [v for v in VARIANT_BLOCKS if RUN_ONLY is None or v in RUN_ONLY or v == 'R1']
_FRAMES2 = {}


def models_for(variant):
    return MODELS_FOR.get(variant, MODELS)


def frames2(variant, model):
    """(train, test, 类别列)。R1 = F0 + P + T + M；C 块只加给 CatBoost。"""
    blocks = VARIANT_BLOCKS[variant]
    kind = 'str' if model == 'catboost' else 'category'
    key = (tuple(blocks), kind)
    if key not in _FRAMES2:
        nums = ([both[RAW_NUM], F0_EXTRA] + [BLOCKS[b] for b in R1_BLOCKS]
                + [BLOCKS2[b] for b in blocks if b in BLOCKS2])
        cats = [both[CAT]] + ([CB_ONLY[b] for b in blocks if b in CB_ONLY] if kind == 'str' else [])
        frame = pd.concat(nums + cats, axis=1)
        require(frame.columns.is_unique, f'{variant} 列名重复')
        cat_cols = [c for part in cats for c in part.columns]
        if kind == 'str':
            frame[cat_cols] = frame[cat_cols].astype(str).astype(object)
        else:
            frame[cat_cols] = frame[cat_cols].astype('category')
        _FRAMES2[key] = (frame.iloc[:n_train].reset_index(drop=True),
                         frame.iloc[n_train:].reset_index(drop=True), cat_cols)
    return _FRAMES2[key]


new_cols = pd.concat([M2, D, H, S], axis=1)
display(pd.DataFrame({'train_mean': new_cols.iloc[:n_train].mean(), 'test_mean': new_cols.iloc[n_train:].mean(),
                      'nan_share': new_cols.isna().mean()}).round(4))
print('C 块类别数：', {c: int(C[c].nunique()) for c in C.columns})
print('各变体列数（LightGBM / XGBoost）：', {v: frames2(v, 'lgb')[0].shape[1] for v in VARIANT_BLOCKS if v != 'C'},
      '｜CatBoost C：', frames2('C', 'catboost')[0].shape[1])

,train_mean,test_mean,nan_share
meet_n_states,3.2039,3.2371,0.0000
meet_n_countries,2.0094,2.0170,0.0000
meet_n_wclass,10.7194,10.7360,0.0000
meet_share_tested,0.7627,0.7604,0.0000
meet_share_raw,0.6208,0.6212,0.0000
meet_share_multi,0.0234,0.0240,0.0000
meet_share_bw2dec,0.2949,0.2969,0.0000
meet_share_junior,0.4549,0.4538,0.0000
meet_share_master,0.1703,0.1714,0.0000
meet_age_std,10.0628,10.0618,0.0104


C 块类别数： {'meet_id': 10546, 'athlete_id': 46893}
各变体列数（LightGBM / XGBoost）： {'R1': 33, 'M2': 47, 'D': 39, 'H': 41, 'S': 34} ｜CatBoost C： 35


## 5. 第三轮特征与各模型的最终特征表（与 11 相同，作为本轮基线 R3）

In [5]:
_FRAMES2.clear()                    # 09 的变体表不再需要，释放内存
birth_num = np.floor(both['MeetYear'] - both['Age'] + 1e-9).astype(int)


def fuzzy_count(prefix_key):
    """同前缀、出生年为 b−1、b、b+1 的行数（整数年龄推算的出生年可能差 1）。"""
    exact = prefix_key + '|' + birth_num.astype(str)
    table = exact.value_counts()
    total = exact.map(table).astype('float64')
    for d in (-1, 1):
        total = total + (prefix_key + '|' + (birth_num + d).astype(str)).map(table).fillna(0)
    return total


# H2：运动员生涯（athlete、valid 来自 09 的特征单元；M、T 来自 07；M2 来自 09）
bw = both['BodyweightKg']
H2 = pd.DataFrame({
    'ath_max_meet_states': M2['meet_n_states'].groupby(athlete).transform('max'),
    'ath_mean_meet_states': M2['meet_n_states'].groupby(athlete).transform('mean'),
    'ath_max_meet_n': M['meet_n'].groupby(athlete).transform('max'),
    'ath_max_nat_states': M2['nat_n_states'].groupby(athlete).transform('max'),
    'ath_ever_abroad': T['abroad'].groupby(athlete).transform('max'),
    'ath_n_wclass': n_distinct(both['WeightClassKg'], athlete),
    'ath_bw_range': bw.groupby(athlete).transform('max') - bw.groupby(athlete).transform('min'),
    'ath_first_age': both['Age'].groupby(athlete).transform('min'),
    'ath_n_same_year': both.groupby([athlete, both['MeetYear']])['Age'].transform('size').astype('float64'),
    'ath_share_tested': (both['Tested'] == 'Yes').astype('float64').groupby(athlete).transform('mean'),
    'ath_n_fuzzy': fuzzy_count(sex + '|' + both['Country'] + '|' + both['State'] + '|' + both['Federation']),
    'ath0_n_fuzzy': fuzzy_count(sex + '|' + both['Country'] + '|' + both['State']),
})
H2.loc[~valid] = np.nan

# E：比赛的经验结构
experience = H['ath_years_since_first']
E = pd.DataFrame({
    'meet_mean_ath_n': H['ath_n'].groupby(meet).transform('mean'),
    'meet_mean_ath_exp': experience.groupby(meet).transform('mean'),
    'meet_share_veterans': (experience >= 3).astype('float64').where(experience.notna()).groupby(meet).transform('mean'),
    'meet_share_newcomers': (H['ath_n'] == 1).astype('float64').where(H['ath_n'].notna()).groupby(meet).transform('mean'),
})

ROUND3 = {'H2': H2, 'E': E}
for name, frame in ROUND3.items():
    require(len(frame) == len(both) and frame.index.equals(both.index), f'{name} 行数或索引不符')
    require(not np.isinf(frame.to_numpy(dtype='float64')).any(), f'{name} 含无穷大')
NUMERIC_BLOCKS = {**BLOCKS, **BLOCKS2, **ROUND3}          # P T M（07）、M2 D H S（09）、H2 E（本轮）
R2_BLOCKS = ['P', 'T', 'M', 'H']                          # 09 的 ALL2
EXTRAS = {'lgb': [], 'xgb': ['M2', 'S', 'D'], 'catboost': ['M2', 'C']}   # PM：09 中各模型单独有效的块
VARIANTS3 = {'R2': [], 'H2': ['H2'], 'E': ['E'], 'PM': ['PM']}
SCREEN3 = [v for v in VARIANTS3 if RUN_ONLY is None or v in RUN_ONLY or v == 'R2']
_FRAMES3 = {}


def blocks_for(variant, model):
    out = list(R2_BLOCKS)
    for b in VARIANTS3[variant]:
        out += EXTRAS[model] if b == 'PM' else [b]
    return out


def runs(model, variant):
    """该模型在这个变体上的特征与 R2 不同才需要训练（例如 PM 对 LightGBM 没有变化）。"""
    return variant == 'R2' or blocks_for(variant, model) != blocks_for('R2', model)


def frames3(variant, model):
    """(train, test, 类别列)。"""
    blocks = blocks_for(variant, model)
    kind = 'str' if model == 'catboost' else 'category'
    key = (tuple(blocks), kind)
    if key not in _FRAMES3:
        nums = [both[RAW_NUM], F0_EXTRA] + [NUMERIC_BLOCKS[b] for b in blocks if b in NUMERIC_BLOCKS]
        cats = [both[CAT]] + ([CB_ONLY[b] for b in blocks if b in CB_ONLY] if kind == 'str' else [])
        frame = pd.concat(nums + cats, axis=1)
        require(frame.columns.is_unique, f'{variant} {model} 列名重复')
        cat_cols = [c for part in cats for c in part.columns]
        if kind == 'str':
            frame[cat_cols] = frame[cat_cols].astype(str).astype(object)
        else:
            frame[cat_cols] = frame[cat_cols].astype('category')
        _FRAMES3[key] = (frame.iloc[:n_train].reset_index(drop=True),
                         frame.iloc[n_train:].reset_index(drop=True), cat_cols)
    return _FRAMES3[key]


new_cols = pd.concat([H2, E], axis=1)
display(pd.DataFrame({'train_mean': new_cols.iloc[:n_train].mean(), 'test_mean': new_cols.iloc[n_train:].mean(),
                      'nan_share': new_cols.isna().mean()}).round(4))
print({f'{v}/{m}': frames3(v, m)[0].shape[1] for v in VARIANTS3 for m in MODELS if runs(m, v)})

# ---- 最终特征：10 的 ALL3（三个块都达标）----
_FRAMES3.clear()
VARIANTS3['ALL3'] = ['H2', 'E', 'PM']
FAMILY = {'lgb': 'lgb', 'lgb_rel': 'lgb', 'tabm': 'lgb', 'xgb': 'xgb', 'catboost': 'catboost', 'cb_rel': 'catboost'}
FRAMES = {m: frames3('ALL3', FAMILY[m]) for m in FINAL_MODELS}
for m, (Xtr_m, Xte_m, cats_m) in FRAMES.items():
    print(f'{m:9s} {Xtr_m.shape[1]} 列（{len(cats_m)} 个类别列），块：{blocks_for("ALL3", FAMILY[m])}')

,train_mean,test_mean,nan_share
ath_max_meet_states,7.2553,7.2965,0.0184
ath_mean_meet_states,3.2682,3.2702,0.0184
ath_max_meet_n,112.4144,112.6788,0.0184
ath_max_nat_states,17.7174,17.7553,0.0184
ath_ever_abroad,0.1208,0.1206,0.0184
ath_n_wclass,6.3779,6.4201,0.0184
ath_bw_range,48.6372,49.1032,0.0184
ath_first_age,24.1705,24.0925,0.0184
ath_n_same_year,4.9418,4.9522,0.0184
ath_share_tested,0.7628,0.7612,0.0184


{'R2/lgb': 41, 'R2/xgb': 41, 'R2/catboost': 41, 'H2/lgb': 53, 'H2/xgb': 53, 'H2/catboost': 53, 'E/lgb': 45, 'E/xgb': 45, 'E/catboost': 45, 'PM/xgb': 62, 'PM/catboost': 57}
lgb       57 列（13 个类别列），块：['P', 'T', 'M', 'H', 'H2', 'E']
xgb       78 列（13 个类别列），块：['P', 'T', 'M', 'H', 'H2', 'E', 'M2', 'S', 'D']
lgb_rel   57 列（13 个类别列），块：['P', 'T', 'M', 'H', 'H2', 'E']
catboost  73 列（15 个类别列），块：['P', 'T', 'M', 'H', 'H2', 'E', 'M2', 'C']
cb_rel    73 列（15 个类别列），块：['P', 'T', 'M', 'H', 'H2', 'E', 'M2', 'C']
tabm      57 列（13 个类别列），块：['P', 'T', 'M', 'H', 'H2', 'E']


## 6. `id` 结构检查（只用 CPU，几秒）

In [6]:
from scipy.stats import spearmanr

ids_all = pd.concat([train['id'], test['id']], ignore_index=True).to_numpy()
order = np.argsort(ids_all)
perm = np.random.default_rng(0).permutation(len(ids_all))
checks = {'同一场比赛': meet.astype(str).to_numpy(), '同一联盟': both['Federation'].astype(str).to_numpy(),
          '同一比赛年份': both['MeetYear'].to_numpy(), '同一国家': both['Country'].astype(str).to_numpy()}
rows = []
for name, values in checks.items():
    by_id = float((values[order][1:] == values[order][:-1]).mean())
    random = float((values[perm][1:] == values[perm][:-1]).mean())
    rows.append({'相邻两行': name, '按 id 排序': by_id, '随机顺序': random, '倍数': by_id / max(random, 1e-12)})
id_table = pd.DataFrame(rows).set_index('相邻两行')
display(id_table.round(4))
rho = float(spearmanr(ids_all, both['MeetYear'].to_numpy())[0])
print(f'id 与比赛年份的 Spearman 相关：{rho:.4f}')
ID_STRUCTURE = bool(id_table.loc['同一场比赛', '倍数'] >= 1.5 or abs(rho) >= 0.1)
print('id 含有比赛结构：', ID_STRUCTURE, '→', '加入 ID 变体' if ID_STRUCTURE else '不测试 id（顺序是随机的）')
id_table.to_csv(OUTPUT_DIR / 'summary_id_check.csv')

,按 id 排序,随机顺序,倍数
相邻两行,,,
同一场比赛,0.0004,0.0004,0.9683
同一联盟,0.0579,0.0582,0.9951
同一比赛年份,0.0589,0.0579,1.0181
同一国家,0.2289,0.2309,0.9914


id 与比赛年份的 Spearman 相关：0.0021
id 含有比赛结构： False → 不测试 id（顺序是随机的）


## 7. 基于成绩的嵌套特征（AS 运动员实力、MS 比赛实力）

In [7]:
N_ALL = len(both)
TEST_ROWS = np.arange(n_train, N_ALL)
REL_ALL = np.full(N_ALL, np.nan)
REL_ALL[:n_train] = y / dots_scale(train['Sex'], train['BodyweightKg'])    # 相对力量（DOTS 分）
TOTAL_ALL = np.full(N_ALL, np.nan)
TOTAL_ALL[:n_train] = y
BIRTH_ALL = pd.Series(np.floor(both['MeetYear'] - both['Age'] + 1e-9).astype(int).to_numpy())
YEAR_ALL = pd.Series(both['MeetYear'].astype(int).to_numpy())
BW_ALL = both['BodyweightKg'].to_numpy(dtype='float64')
VALID_ALL = (both['Country'] != 'Unknown').to_numpy()


def text_key_all(columns):
    out = both[columns[0]].astype(str)
    for c in columns[1:]:
        out = out + '|' + both[c].astype(str)
    return out.reset_index(drop=True)


PFX_ALL = {'A0': text_key_all(['Sex', 'Country', 'State']),
           'A1': text_key_all(['Sex', 'Country', 'State', 'Federation']),
           'A2': text_key_all(['Sex', 'Country', 'State', 'Federation', 'Equipment'])}
KB_SHIFT = {k: {d: p + '|' + (BIRTH_ALL + d).astype(str) for d in (-1, 0, 1)} for k, p in PFX_ALL.items()}
KB_A1 = KB_SHIFT['A1'][0]
KBY_A1 = KB_A1 + '|' + YEAR_ALL.astype(str)
DECAY_GAPS = range(-8, 9)
KBY_SHIFT = {d: KB_A1 + '|' + (YEAR_ALL + d).astype(str) for d in DECAY_GAPS}
MEET_ALL = {'meet': meet.reset_index(drop=True).astype(str),
            'div': text_key_all(['Federation', 'MeetYear', 'MeetCountry', 'MeetState', 'Sex', 'Equipment'])}
EXACT_ALL = pd.Series(pd.util.hash_pandas_object(both[RAW_FEATURES], index=False).to_numpy()).astype(str)
CORE_ALL = pd.Series(pd.util.hash_pandas_object(both[CORE9], index=False).to_numpy()).astype(str)


def agg(keys, values, ref, funcs=('sum', 'count', 'max')):
    frame = pd.DataFrame({'k': keys.iloc[ref].to_numpy(), 'v': values[ref]})
    return frame.groupby('k')['v'].agg(list(funcs))


def lookup(table, column, keys, fill):
    return keys.map(table[column]).fillna(fill).to_numpy(dtype='float64')


def athlete_block(ref, query):
    """ref：有成绩的参考行（训练行号）；query：要算特征的行（both 中的行号，不在 ref 里）。"""
    cols, m = {}, len(query)
    for k in ['A0', 'A1', 'A2']:
        table = agg(KB_SHIFT[k][0], REL_ALL, ref)
        bw_table = agg(KB_SHIFT[k][0], BW_ALL, ref, ('sum',))
        s, n, bws, mx = np.zeros(m), np.zeros(m), np.zeros(m), np.full(m, -np.inf)
        for d in (-1, 0, 1):                                       # 出生年 ±1
            kq = KB_SHIFT[k][d].iloc[query]
            s, n = s + lookup(table, 'sum', kq, 0), n + lookup(table, 'count', kq, 0)
            mx = np.maximum(mx, lookup(table, 'max', kq, -np.inf))
            bws = bws + lookup(bw_table, 'sum', kq, 0)
        has = n > 0
        cols[f'as_{k}_n'] = n
        cols[f'as_{k}_mean'] = np.where(has, s / np.maximum(n, 1), np.nan)
        cols[f'as_{k}_max'] = np.where(has, mx, np.nan)
        if k == 'A1':
            cols['as_A1_bw_diff'] = np.where(has, BW_ALL[query] - bws / np.maximum(n, 1), np.nan)
    table = agg(KBY_A1, REL_ALL, ref, ('sum', 'count'))          # 时间衰减与年份差（A1，精确出生年）
    ws, wn, gap_sum, gap_n, gap_min = np.zeros(m), np.zeros(m), np.zeros(m), np.zeros(m), np.full(m, np.inf)
    for d in DECAY_GAPS:
        kq = KBY_SHIFT[d].iloc[query]
        sd, nd = lookup(table, 'sum', kq, 0), lookup(table, 'count', kq, 0)
        w = float(np.exp(-abs(d) / 3.0))
        ws, wn = ws + w * sd, wn + w * nd
        gap_sum, gap_n = gap_sum + abs(d) * nd, gap_n + nd
        gap_min = np.where(nd > 0, np.minimum(gap_min, abs(d)), gap_min)
    cols['as_A1_decay'] = np.where(wn > 0, ws / np.maximum(wn, 1e-12), np.nan)
    cols['as_A1_gap_mean'] = np.where(gap_n > 0, gap_sum / np.maximum(gap_n, 1), np.nan)
    cols['as_A1_gap_min'] = np.where(np.isfinite(gap_min), gap_min, np.nan)
    cols['as_same_equipment_share'] = np.where(cols['as_A1_n'] > 0,
                                               cols['as_A2_n'] / np.maximum(cols['as_A1_n'], 1), np.nan)
    frame = pd.DataFrame(cols)
    frame.loc[~VALID_ALL[query]] = np.nan                           # 国家未知时不做运动员匹配
    for name, keys in [('tw_exact', EXACT_ALL), ('tw_core', CORE_ALL)]:
        t = agg(keys, TOTAL_ALL, ref, ('sum', 'count'))
        kq = keys.iloc[query]
        s, n = lookup(t, 'sum', kq, 0), lookup(t, 'count', kq, 0)
        frame[f'{name}_total'] = np.where(n > 0, s / np.maximum(n, 1), np.nan)
        frame[f'{name}_n'] = n
    return frame


def meet_block(ref, query):
    cols = {}
    for name, keys in MEET_ALL.items():
        t = agg(keys, REL_ALL, ref)
        kq = keys.iloc[query]
        s, n, mx = lookup(t, 'sum', kq, 0), lookup(t, 'count', kq, 0), lookup(t, 'max', kq, np.nan)
        cols[f'ms_{name}_n'] = n
        cols[f'ms_{name}_mean'] = np.where(n > 0, s / np.maximum(n, 1), np.nan)
        cols[f'ms_{name}_max'] = np.where(n > 0, mx, np.nan)
    return pd.DataFrame(cols)


BLOCK_FUNCS = {'AS': athlete_block, 'MS': meet_block}
_FOLD_FEATS = {}


def nested_features(block, fold):
    """(训练行特征 n_train 行, 测试行特征)：外层训练部分用内层折外，外层验证行和测试行用整个训练部分。"""
    key = (block, fold)
    if key not in _FOLD_FEATS:
        fn = BLOCK_FUNCS[block]
        outer_train, outer_valid = np.flatnonzero(FOLD_IDS != fold), np.flatnonzero(FOLD_IDS == fold)
        parts = []
        inner = KFold(INNER_FEATURE_FOLDS, shuffle=True, random_state=100 + fold)
        for inner_ref, inner_query in inner.split(outer_train):
            part = fn(outer_train[inner_ref], outer_train[inner_query])
            part.index = outer_train[inner_query]
            parts.append(part)
        part = fn(outer_train, outer_valid)
        part.index = outer_valid
        parts.append(part)
        train_part = pd.concat(parts).sort_index()
        require(train_part.index.equals(pd.RangeIndex(n_train)), f'{block} 第 {fold} 折的嵌套特征行数不符')
        test_part = fn(outer_train, TEST_ROWS).reset_index(drop=True)
        _FOLD_FEATS[key] = (train_part, test_part)
    return _FOLD_FEATS[key]


ID_TRAIN = pd.DataFrame({'row_id': train['id'].to_numpy(dtype='float64')})
ID_TEST = pd.DataFrame({'row_id': test['id'].to_numpy(dtype='float64')})
VARIANTS4 = {'R3': [], 'AS': ['AS'], 'MS': ['MS']}
if ID_STRUCTURE:
    VARIANTS4['ID'] = ['ID']
FAMILY_OF = {'lgb': 'lgb', 'xgb': 'xgb', 'cb_rel': 'cb_rel', 'cb_allo': 'cb_rel', 'cb_gl': 'cb_rel'}


def frames_aug(variant, model, fold):
    base_tr, base_te, _ = FRAMES[FAMILY_OF[model]]
    blocks = VARIANTS4[variant]
    extra_tr = [nested_features(b, fold)[0] for b in blocks if b in BLOCK_FUNCS]
    extra_te = [nested_features(b, fold)[1] for b in blocks if b in BLOCK_FUNCS]
    if 'ID' in blocks:
        extra_tr.append(ID_TRAIN)
        extra_te.append(ID_TEST)
    if not extra_tr:
        return base_tr, base_te
    return pd.concat([base_tr] + extra_tr, axis=1), pd.concat([base_te] + extra_te, axis=1)


started = time.perf_counter()
demo_tr, demo_te = nested_features('AS', 1)
print(f'AS 第 1 折：{demo_tr.shape[1]} 列，用时 {time.perf_counter() - started:.0f}s')
display(pd.DataFrame({'train_nonnull': demo_tr.notna().mean(), 'test_nonnull': demo_te.notna().mean(),
                      'train_mean': demo_tr.mean(), 'test_mean': demo_te.mean()}).round(3))
corr = pd.Series({c: np.corrcoef(demo_tr[c].fillna(demo_tr[c].median()), y)[0, 1] for c in demo_tr.columns})
print('与 TotalKg 的相关（第 1 折，训练行为嵌套折外值）：'); display(corr.round(3).sort_values())

AS 第 1 折：18 列，用时 5s


,train_nonnull,test_nonnull,train_mean,test_mean
as_A0_n,0.981,0.983,113.409,137.562
as_A0_mean,0.961,0.967,363.942,364.205
as_A0_max,0.961,0.967,519.810,525.192
as_A1_n,0.981,0.983,45.641,54.747
as_A1_mean,0.862,0.883,364.316,364.492
as_A1_max,0.862,0.883,472.560,476.486
as_A1_bw_diff,0.862,0.883,0.027,0.063
as_A2_n,0.981,0.983,31.710,37.871
as_A2_mean,0.820,0.842,363.503,364.033
as_A2_max,0.820,0.842,456.914,460.707


与 TotalKg 的相关（第 1 折，训练行为嵌套折外值）：


,0
as_same_equipment_share,-0.070
tw_exact_n,-0.005
tw_core_n,-0.005
as_A1_gap_min,0.008
as_A1_n,0.057
as_A2_n,0.064
tw_exact_total,0.078
as_A1_gap_mean,0.078
tw_core_total,0.082
as_A0_n,0.145


## 8. 模型定义（与 11 相同）+ 两种新的力量目标 + 预检

- `cb_allo`：目标 = 总成绩 ÷ (体重 / 80)^(2/3)（异速生长）。
- `cb_gl`：目标 = IPF GL 分，系数按性别 × 是否装备（Raw / Wraps 用无装备系数，Single-ply / Multi-ply / Unlimited 用装备系数）。
- 两者都按系数² 加权（与 cb_rel 相同），所以仍在最小化 kg 的 RMSE；参数与 cb_rel 相同（CatBoost C2）。

In [8]:
LGB_PARAMS = dict(learning_rate=0.05, num_leaves=63, min_child_samples=40, subsample=0.8, subsample_freq=1,
                  colsample_bytree=0.8, cat_smooth=20, verbose=-1)
XGB_PARAMS = dict(learning_rate=0.03, max_depth=8, min_child_weight=10.0, subsample=0.8, colsample_bytree=0.7,
                  reg_lambda=1.0, tree_method='hist', device='cuda', enable_categorical=True,
                  max_cat_to_onehot=1, eval_metric='rmse')
CB_PARAMS = dict(learning_rate=0.05, depth=8, l2_leaf_reg=10.0, bootstrap_type='Bernoulli', subsample=0.8,
                 border_count=128)
TABM_PARAMS = dict(device='cuda', n_cv=1, n_refit=0, verbosity=0, num_emb_type='pwl')
TABM_FALLBACK = dict(device='cuda', n_cv=1, n_refit=0, verbosity=0)
PATIENCE = {'lgb': 200, 'lgb_rel': 200, 'xgb': 200, 'catboost': 300, 'cb_rel': 300}
GBDT = [m for m in FINAL_MODELS if m != 'tabm']

SCALE = dots_scale(train['Sex'], train['BodyweightKg'])           # dots_scale 来自 09 的特征单元
SCALE_TEST = dots_scale(test['Sex'], test['BodyweightKg'])
require(min(SCALE.min(), SCALE_TEST.min()) > 0.3, 'DOTS 系数异常')

X_NN, XT_NN, CATS_NN = FRAMES['tabm']
NUM_NN = [c for c in X_NN.columns if c not in CATS_NN]
NAN_NN = [c for c in NUM_NN if X_NN[c].isna().any() or XT_NN[c].isna().any()]


def target_and_weight(model, rows):
    if model.endswith('_rel'):
        return y[rows] / SCALE[rows], SCALE[rows] ** 2       # 相对力量误差 × scale = kg 误差
    return y[rows], None


def gbdt_fit(model, seed, Xa, ya, wa, Xs=None, ys=None, ws=None, rounds=MAX_ROUNDS):
    """Xs 为 None 时不早停（全量重训）。返回 (模型, 最佳轮数)。"""
    cats = FRAMES[model][2]
    if model in ('lgb', 'lgb_rel'):
        m = lgb.LGBMRegressor(n_estimators=rounds, random_state=seed, n_jobs=-1, **LGB_PARAMS)
        if Xs is None:
            m.fit(Xa, ya, sample_weight=wa)
            return m, rounds
        m.fit(Xa, ya, sample_weight=wa, eval_set=[(Xs, ys)], eval_sample_weight=None if ws is None else [ws],
              callbacks=[lgb.early_stopping(PATIENCE[model], verbose=False)])
        return m, int(m.best_iteration_ or rounds)
    if model == 'xgb':
        if Xs is None:
            m = xgb.XGBRegressor(n_estimators=rounds, random_state=seed, **XGB_PARAMS)
            m.fit(Xa, ya, verbose=False)
            return m, rounds
        m = xgb.XGBRegressor(n_estimators=rounds, random_state=seed, early_stopping_rounds=PATIENCE['xgb'],
                             **XGB_PARAMS)
        m.fit(Xa, ya, eval_set=[(Xs, ys)], verbose=False)
        return m, int(m.best_iteration) + 1
    common = dict(iterations=rounds, loss_function='RMSE', eval_metric='RMSE', random_seed=seed, verbose=0,
                  task_type='GPU', devices='0', allow_writing_files=False, **CB_PARAMS)
    if Xs is None:
        m = catboost.CatBoostRegressor(**common)
        m.fit(catboost.Pool(Xa, ya, cat_features=cats, weight=wa))
        return m, rounds
    m = catboost.CatBoostRegressor(od_type='Iter', od_wait=PATIENCE[model], **common)
    m.fit(catboost.Pool(Xa, ya, cat_features=cats, weight=wa),
          eval_set=catboost.Pool(Xs, ys, cat_features=cats, weight=ws), use_best_model=True)
    return m, int(m.get_best_iteration()) + 1


def gbdt_predict(model, m, best, X):
    if model in ('lgb', 'lgb_rel'):
        return m.predict(X, num_iteration=best)
    if model == 'xgb':
        return m.predict(X, iteration_range=(0, best))
    return m.predict(X)


def nn_frames(*frames):
    """数值缺失用第一个表（外层训练部分）的中位数填补并加缺失标记；类别保持 category。"""
    median = frames[0][NUM_NN].median()
    out = []
    for frame in frames:
        z = frame.copy()
        for c in NAN_NN:
            z[f'{c}_isna'] = z[c].isna().astype('float64')
        z[NUM_NN] = z[NUM_NN].fillna(median).fillna(0.0)
        out.append(z.reset_index(drop=True))
    return out


def tabm_fit_predict(seed, a, s, v, Xt, overrides=None):
    require(TabM_D_Regressor is not None, f'pytabkit 不可用：{PYTABKIT_ERROR}')
    params = dict(TABM_PARAMS, random_state=seed, **(overrides or {}))
    unknown = set(params) - set(inspect.signature(TabM_D_Regressor.__init__).parameters)
    require(not unknown, f'TabM 不支持参数 {sorted(unknown)}')
    Za, Zs, Zv, Zt = nn_frames(X_NN.iloc[a], X_NN.iloc[s], X_NN.iloc[v], Xt)
    mu, sd = y[a].mean(), y[a].std()
    m = TabM_D_Regressor(**params)
    m.fit(Za, (y[a] - mu) / sd, Zs, (y[s] - mu) / sd, cat_col_names=[c for c in Za.columns if c in CATS_NN])
    p_val = np.asarray(m.predict(Zv), dtype='float64').reshape(-1) * sd + mu
    p_test = np.asarray(m.predict(Zt), dtype='float64').reshape(-1) * sd + mu
    del m
    gc.collect()
    torch.cuda.empty_cache()
    return p_val, p_test


def fit_fold(model, seed, a, s, v, test_rows=None, rounds=MAX_ROUNDS, overrides=None):
    """a / s / v：内层拟合 / 内层早停 / 外层验证 的训练行号。返回 (验证预测, 测试预测, 最佳轮数或 None)。"""
    Xtr, Xte, _ = FRAMES[model]
    Xt = Xte if test_rows is None else Xte.iloc[test_rows]
    scale_t = SCALE_TEST if test_rows is None else SCALE_TEST[test_rows]
    if model == 'tabm':
        p_val, p_test = tabm_fit_predict(seed, a, s, v, Xt, overrides)
        return p_val, p_test, None
    ya, wa = target_and_weight(model, a)
    ys, ws = target_and_weight(model, s)
    if wa is not None:
        norm = wa.mean()
        wa, ws = wa / norm, ws / norm
    m, best = gbdt_fit(model, seed, Xtr.iloc[a], ya, wa, Xtr.iloc[s], ys, ws, rounds)
    p_val, p_test = gbdt_predict(model, m, best, Xtr.iloc[v]), gbdt_predict(model, m, best, Xt)
    if model.endswith('_rel'):
        p_val, p_test = np.asarray(p_val) * SCALE[v], np.asarray(p_test) * scale_t
    return np.asarray(p_val, dtype='float64'), np.asarray(p_test, dtype='float64'), best


def fit_full(model, seed, rounds, test_rows=None):
    """全部训练行、固定轮数、不早停。返回测试预测。"""
    Xtr, Xte, _ = FRAMES[model]
    Xt = Xte if test_rows is None else Xte.iloc[test_rows]
    scale_t = SCALE_TEST if test_rows is None else SCALE_TEST[test_rows]
    rows = np.arange(n_train)
    ya, wa = target_and_weight(model, rows)
    if wa is not None:
        wa = wa / wa.mean()
    m, _ = gbdt_fit(model, seed, Xtr, ya, wa, rounds=rounds)
    p_test = gbdt_predict(model, m, rounds, Xt)
    if model.endswith('_rel'):
        p_test = np.asarray(p_test) * scale_t
    return np.asarray(p_test, dtype='float64')




def gl_scale(sex_values, equipment_values, bw_values):
    sex = np.asarray(sex_values).astype(str)
    equipped = ~np.isin(np.asarray(equipment_values).astype(str), ['Raw', 'Wraps'])
    bw = np.maximum(np.asarray(bw_values, dtype='float64'), 30.0)
    coef = {('M', False): (1199.72839, 1025.18162, 0.00921), ('M', True): (1236.25115, 1449.21864, 0.01644),
            ('F', False): (610.32796, 1045.59282, 0.03048), ('F', True): (758.63878, 949.31382, 0.02435)}

    def curve(s, e):
        a, b, c = coef[(s, e)]
        return a - b * np.exp(-c * bw)

    male = np.where(equipped, curve('M', True), curve('M', False))
    female = np.where(equipped, curve('F', True), curve('F', False))
    return np.where(sex == 'M', male, np.where(sex == 'F', female, 0.5 * (male + female))) / 100.0


ALLO = (train['BodyweightKg'].to_numpy(dtype='float64') / 80.0) ** (2 / 3)
ALLO_TEST = (test['BodyweightKg'].to_numpy(dtype='float64') / 80.0) ** (2 / 3)
GL = gl_scale(train['Sex'], train['Equipment'], train['BodyweightKg'])
GL_TEST = gl_scale(test['Sex'], test['Equipment'], test['BodyweightKg'])
require(min(GL.min(), GL_TEST.min()) > 0 and min(ALLO.min(), ALLO_TEST.min()) > 0, '力量系数异常')
SCALE_OF = {'cb_rel': (SCALE, SCALE_TEST), 'cb_allo': (ALLO, ALLO_TEST), 'cb_gl': (GL, GL_TEST)}
for extra in TARGET_MODELS:
    FRAMES[extra] = FRAMES['cb_rel']
    PATIENCE[extra] = PATIENCE['cb_rel']


def fit_matrix(model, Xtr, Xte, a, s, v, seed=SEED, rounds=MAX_ROUNDS, test_rows=None):
    """用给定的特征表训练（内层拟合 a、早停 s），返回 (外层验证预测, 测试预测, 最佳轮数)。"""
    Xt = Xte if test_rows is None else Xte.iloc[test_rows]
    if model in SCALE_OF:
        scale, scale_test = SCALE_OF[model]
        scale_t = scale_test if test_rows is None else scale_test[test_rows]
        ya, ys = y[a] / scale[a], y[s] / scale[s]
        norm = (scale[a] ** 2).mean()
        wa, ws = scale[a] ** 2 / norm, scale[s] ** 2 / norm
    else:
        ya, ys, wa, ws = y[a], y[s], None, None
    m, best = gbdt_fit(model, seed, Xtr.iloc[a], ya, wa, Xtr.iloc[s], ys, ws, rounds)
    p_val = np.asarray(gbdt_predict(model, m, best, Xtr.iloc[v]), dtype='float64')
    p_test = np.asarray(gbdt_predict(model, m, best, Xt), dtype='float64')
    if model in SCALE_OF:
        p_val, p_test = p_val * scale[v], p_test * scale_t
    return p_val, p_test, best


rng = np.random.default_rng(SEED)
probe = rng.choice(np.flatnonzero(FOLD_IDS != 1), 3000, replace=False)
pa, ps, pv = probe[:2000], probe[2000:2500], probe[2500:]
failed = {}
checks = [(m, v) for v in VARIANTS4 for m in SCREEN_MODELS] + [(m, 'R3') for m in TARGET_MODELS]
for model, variant in checks:
    try:
        Xtr, Xte = frames_aug(variant, model, 1)
        p_val, p_test, _ = fit_matrix(model, Xtr, Xte, pa, ps, pv, rounds=30, test_rows=np.arange(300))
        require(np.isfinite(p_val).all() and np.isfinite(p_test).all(), '预测含非有限值')
        print(f'预检通过：{model:8s} {variant:3s}（{Xtr.shape[1]} 列）RMSE@30轮 {rmse(y[pv], p_val):.1f}')
    except Exception as exc:
        failed[(model, variant)] = repr(exc)
        print(f'预检失败，跳过：{model} {variant} → {exc!r}')
require(not any(v == 'R3' and m in SCREEN_MODELS for (m, v) in failed), '基线 R3 预检失败，请检查环境')

预检通过：lgb      R3 （57 列）RMSE@30轮 99.3
预检通过：xgb      R3 （78 列）RMSE@30轮 122.8
预检通过：cb_rel   R3 （73 列）RMSE@30轮 93.2
预检通过：lgb      AS （75 列）RMSE@30轮 98.5
预检通过：xgb      AS （96 列）RMSE@30轮 122.3
预检通过：cb_rel   AS （91 列）RMSE@30轮 92.8
预检通过：lgb      MS （63 列）RMSE@30轮 97.2
预检通过：xgb      MS （84 列）RMSE@30轮 122.4
预检通过：cb_rel   MS （79 列）RMSE@30轮 91.7
预检通过：cb_allo  R3 （73 列）RMSE@30轮 95.3
预检通过：cb_gl    R3 （73 列）RMSE@30轮 94.3


## 9. 筛选：R3 与每个新块单独加入（五折，可续跑）

In [9]:
def run_one(model, variant, fold):
    stem = f'{model}__{variant}__fold{fold}'
    json_path, npz_path = OUTPUT_DIR / f'{stem}.json', OUTPUT_DIR / f'{stem}.npz'
    if json_path.is_file() and npz_path.is_file():
        return json.loads(json_path.read_text(encoding='utf-8')), 'cache'
    Xtr, Xte = frames_aug(variant, model, fold)
    fit_idx, valid_idx = np.flatnonzero(FOLD_IDS != fold), np.flatnonzero(FOLD_IDS == fold)
    inner_fit, inner_stop = train_test_split(fit_idx, test_size=INNER_STOP_FRACTION, random_state=SEED)  # = 07–13
    started = time.perf_counter()
    p_val, p_test, best = fit_matrix(model, Xtr, Xte, inner_fit, inner_stop, valid_idx)
    require(np.isfinite(p_val).all() and np.isfinite(p_test).all(), f'{stem} 预测含非有限值')
    np.savez(npz_path, valid_idx=valid_idx, val=p_val, test=p_test)
    info = {'model': model, 'variant': variant, 'fold': fold, 'blocks': VARIANTS4[variant],
            'n_features': int(Xtr.shape[1]), 'best_iteration': best, 'rmse': rmse(y[valid_idx], p_val),
            'seconds': time.perf_counter() - started, 'gpu': GPU_NAME,
            'finished_at_utc': datetime.now(timezone.utc).isoformat()}
    json_path.write_text(json.dumps(info, ensure_ascii=False, indent=1), encoding='utf-8')
    return info, f'{info["seconds"]:.0f}s'


def run_variants(variants, models):
    for fold in range(1, N_FOLDS + 1):
        fold_started = time.perf_counter()
        for model in models:
            for variant in variants:
                if (model, variant) in failed:
                    continue
                info, how = run_one(model, variant, fold)
                print(f'fold {fold}  {model:8s} {variant:4s}  RMSE {info["rmse"]:.3f}  '
                      f'最佳轮数 {info["best_iteration"]:5d}  ({how})')
        print(f'—— fold {fold} 完成，本次运行用时 {(time.perf_counter() - fold_started) / 60:.1f} 分钟')


run_variants(list(VARIANTS4), SCREEN_MODELS)

fold 1  lgb      R3    RMSE 77.787  最佳轮数   663  (26s)
fold 1  lgb      AS    RMSE 77.767  最佳轮数   553  (28s)
fold 1  lgb      MS    RMSE 77.616  最佳轮数   506  (23s)
fold 1  xgb      R3    RMSE 78.328  最佳轮数   594  (12s)
fold 1  xgb      AS    RMSE 78.325  最佳轮数   647  (15s)
fold 1  xgb      MS    RMSE 78.192  最佳轮数   643  (14s)
fold 1  cb_rel   R3    RMSE 77.228  最佳轮数  2475  (101s)
fold 1  cb_rel   AS    RMSE 78.858  最佳轮数  5369  (211s)
fold 1  cb_rel   MS    RMSE 81.069  最佳轮数  4304  (170s)
—— fold 1 完成，本次运行用时 10.0 分钟
fold 2  lgb      R3    RMSE 77.901  最佳轮数   296  (14s)
fold 2  lgb      AS    RMSE 77.700  最佳轮数   410  (21s)
fold 2  lgb      MS    RMSE 77.923  最佳轮数   357  (17s)
fold 2  xgb      R3    RMSE 78.907  最佳轮数   431  (10s)
fold 2  xgb      AS    RMSE 78.586  最佳轮数   504  (13s)
fold 2  xgb      MS    RMSE 78.914  最佳轮数   439  (11s)
fold 2  cb_rel   R3    RMSE 77.232  最佳轮数  2034  (84s)
fold 2  cb_rel   AS    RMSE 77.996  最佳轮数  2444  (99s)
fold 2  cb_rel   MS    RMSE 81.050  最佳轮数  3344  (13

## 10. 与 R3 配对比较（单模型 + 三模型融合），自动合成 ALL4

In [10]:
def load_oof(model, variant):
    oof, test_pred = np.full(n_train, np.nan), np.zeros(len(test))
    for fold in range(1, N_FOLDS + 1):
        path = OUTPUT_DIR / f'{model}__{variant}__fold{fold}.npz'
        if not path.is_file():
            return None
        saved = np.load(path)
        oof[saved['valid_idx']] = saved['val']
        test_pred += saved['test'] / N_FOLDS
    return oof, test_pred


def fold_rmse(pred):
    return pd.Series({k: rmse(y[FOLD_IDS == k], pred[FOLD_IDS == k]) for k in range(1, N_FOLDS + 1)})


def paired_table(preds, base='R3'):
    base_folds = fold_rmse(preds[base])
    rows = []
    for name, pred in preds.items():
        delta = fold_rmse(pred) - base_folds
        rows.append({'variant': name, 'oof_rmse': rmse(y, pred), 'mean_delta': delta.mean(),
                     'folds_better': int((delta < 0).sum()), **{f'd_fold{k}': v for k, v in delta.items()}})
    table = pd.DataFrame(rows).set_index('variant')
    table['keep'] = (table['mean_delta'] <= KEEP_DELTA) & (table['folds_better'] >= 4)
    table.loc[base, 'keep'] = False
    return table.sort_values('oof_rmse')


def blend_variant(variant, models=SCREEN_MODELS):
    loaded = [load_oof(m, variant) for m in models]
    if any(item is None for item in loaded):
        return None
    P_oof = np.column_stack([o for o, _ in loaded])
    P_test = np.column_stack([t for _, t in loaded])
    w, _ = nnls(P_oof, y)
    return {'oof': P_oof @ w, 'test': P_test @ w, 'weights': [round(float(x), 3) for x in w]}


for model in SCREEN_MODELS:
    preds = {v: load_oof(model, v)[0] for v in VARIANTS4 if load_oof(model, v) is not None}
    if 'R3' in preds:
        print(f'\n=== {model}')
        display(paired_table(preds).round(3))
BLENDS = {v: blend_variant(v) for v in VARIANTS4}
BLENDS = {v: b for v, b in BLENDS.items() if b is not None}
require('R3' in BLENDS, 'R3 还没有全部完成')
blend_table = paired_table({v: b['oof'] for v, b in BLENDS.items()})
blend_table['weights_lgb_xgb_cbrel'] = [str(BLENDS[v]['weights']) for v in blend_table.index]
print('\n=== 三模型 NNLS 融合（主要判读依据）')
display(blend_table.round(3))
blend_table.to_csv(OUTPUT_DIR / 'summary_blend_screen.csv')
KEPT4 = [v for v in VARIANTS4 if v != 'R3' and v in blend_table.index and bool(blend_table.loc[v, 'keep'])]
print('达标的新块：', KEPT4 or '无')
if len(KEPT4) >= 2:
    VARIANTS4['ALL4'] = [b for v in KEPT4 for b in VARIANTS4[v]]
    run_variants(['ALL4'], SCREEN_MODELS)
    BLENDS['ALL4'] = blend_variant('ALL4')
    display(paired_table({v: BLENDS[v]['oof'] for v in ['R3'] + KEPT4 + ['ALL4']}).round(3))
FINAL4 = 'ALL4' if 'ALL4' in BLENDS else (KEPT4[0] if KEPT4 else None)
if FINAL4 is None and len(blend_table) > 1:
    FINAL4 = blend_table.index[0] if blend_table.index[0] != 'R3' else blend_table.index[1]
print('与 R3 做净收益比较的变体：', FINAL4)


=== lgb


,oof_rmse,mean_delta,folds_better,d_fold1,d_fold2,d_fold3,d_fold4,d_fold5,keep
variant,,,,,,,,,
AS,77.562,-0.147,5,-0.021,-0.201,-0.252,-0.201,-0.059,True
MS,77.706,-0.003,3,-0.172,0.023,-0.188,-0.073,0.396,False
R3,77.709,0.000,0,0.000,0.000,0.000,0.000,0.000,False



=== xgb


,oof_rmse,mean_delta,folds_better,d_fold1,d_fold2,d_fold3,d_fold4,d_fold5,keep
variant,,,,,,,,,
AS,78.291,-0.167,5,-0.002,-0.321,-0.105,-0.088,-0.316,True
MS,78.421,-0.036,3,-0.136,0.007,-0.090,-0.075,0.113,False
R3,78.458,0.000,0,0.000,0.000,0.000,0.000,0.000,False



=== cb_rel


,oof_rmse,mean_delta,folds_better,d_fold1,d_fold2,d_fold3,d_fold4,d_fold5,keep
variant,,,,,,,,,
R3,77.160,0.000,0,0.000,0.000,0.000,0.000,0.000,False
AS,78.232,1.070,0,1.630,0.764,1.404,0.760,0.791,False
MS,81.047,3.887,0,3.841,3.818,3.581,3.847,4.350,False



=== 三模型 NNLS 融合（主要判读依据）


,oof_rmse,mean_delta,folds_better,d_fold1,d_fold2,d_fold3,d_fold4,d_fold5,keep,weights_lgb_xgb_cbrel
variant,,,,,,,,,,
R3,76.717,0.000,0,0.000,0.000,0.000,0.000,0.000,False,"[0.211, 0.214, 0.574]"
AS,76.856,0.139,0,0.358,0.044,0.124,0.123,0.047,False,"[0.384, 0.253, 0.365]"
MS,77.233,0.516,0,0.459,0.540,0.419,0.454,0.708,False,"[0.524, 0.282, 0.197]"


达标的新块： 无
与 R3 做净收益比较的变体： AS


## 11. 扣除第二层平滑后的净收益（双胞胎 + S3，交叉拟合，与 11 相同的做法）→ 导出提交文件

In [11]:
CORE = CORE9


def row_keys(frame, columns):
    return pd.util.hash_pandas_object(frame[columns], index=False).to_numpy()


def mate_stats(train_keys, query_keys=None):
    table = pd.DataFrame({'k': train_keys, 'y': y}).groupby('k')['y'].agg(['sum', 'count'])
    if query_keys is None:                      # 训练行：留一
        keys = pd.Series(train_keys)
        n = keys.map(table['count']).to_numpy() - 1
        s = keys.map(table['sum']).to_numpy() - y
    else:                                       # 测试行：全部训练行
        keys = pd.Series(query_keys)
        n = keys.map(table['count']).fillna(0).to_numpy()
        s = keys.map(table['sum']).fillna(0).to_numpy()
    return n, np.where(n > 0, s / np.maximum(n, 1), np.nan)


def twins(query_exact=None, query_core=None):
    n_ex, m_ex = mate_stats(row_keys(train, RAW_FEATURES), query_exact)
    n_co, m_co = mate_stats(row_keys(train, CORE), query_core)
    level = np.select([n_ex > 0, n_co > 0], ['exact', 'core'], default='none')
    return level, np.where(level == 'exact', m_ex, m_co)


def fit_alpha(mask, pred, mate):
    d = mate[mask] - pred[mask]
    denom = float(np.sum(d * d))
    return float(np.clip(np.sum((y[mask] - pred[mask]) * d) / denom, 0, 1)) if denom > 0 else 0.0


def apply_twins(pred, mate, level, alphas):
    out = pred.copy()
    for lv, a in alphas.items():
        m = level == lv
        out[m] = pred[m] + a * (mate[m] - pred[m])
    return out


LEVEL, MATE = twins()
T_LEVEL, T_MATE = twins(row_keys(test, RAW_FEATURES), row_keys(test, CORE))
HALF = (pd.util.hash_pandas_object(train[['Sex', 'Country', 'State']], index=False).to_numpy() % 2).astype(int)
TR, TE = train.reset_index(drop=True), test.reset_index(drop=True)


def text_key(frame, columns):
    out = frame[columns[0]].astype(str)
    for c in columns[1:]:
        out = out + '|' + frame[c].astype(str)
    return out


BY_TR = np.floor(TR['MeetYear'] - TR['Age'] + 1e-9).astype(int)
BY_TE = np.floor(TE['MeetYear'] - TE['Age'] + 1e-9).astype(int)
YR_TR, YR_TE = TR['MeetYear'].astype(int), TE['MeetYear'].astype(int)
PREFIX = {'A0': ['Sex', 'Country', 'State'], 'A1': ['Sex', 'Country', 'State', 'Federation'],
          'A2': ['Sex', 'Country', 'State', 'Federation', 'Equipment'],
          'A3': ['Sex', 'Country', 'State', 'Federation', 'Equipment', 'WeightClassKg']}
SHRINK = {'A0': 5.0, 'A1': 5.0, 'A2': 3.0, 'A3': 2.0}
PFX_TR = {k: text_key(TR, c) for k, c in PREFIX.items()}
PFX_TE = {k: text_key(TE, c) for k, c in PREFIX.items()}
MEET_COLS = ['Federation', 'MeetYear', 'MeetCountry', 'MeetState', 'Sex', 'Equipment']
MEET_TR, MEET_TE = text_key(TR, MEET_COLS), text_key(TE, MEET_COLS)
VALID_TR = (TR['Country'] != 'Unknown').to_numpy(dtype='float64')
VALID_TE = (TE['Country'] != 'Unknown').to_numpy(dtype='float64')


def sums(keys, values):
    table = pd.DataFrame({'k': keys.to_numpy(), 'v': values}).groupby('k')['v'].agg(['sum', 'count'])
    return table['sum'], table['count']


def look(table, keys):
    s, n = table
    return keys.map(s).fillna(0).to_numpy(dtype='float64'), keys.map(n).fillna(0).to_numpy(dtype='float64')


def f_exact(k, r):
    ktr, kte = PFX_TR[k] + '|' + BY_TR.astype(str), PFX_TE[k] + '|' + BY_TE.astype(str)
    table = sums(ktr, r)
    s, n = look(table, ktr)
    st, nt = look(table, kte)
    return (s - r) / (n - 1 + SHRINK[k]), st / (nt + SHRINK[k])


def f_fuzzy(k, r):
    table = sums(PFX_TR[k] + '|' + BY_TR.astype(str), r)
    s, n, st, nt = np.zeros(n_train), np.zeros(n_train), np.zeros(len(TE)), np.zeros(len(TE))
    for d in (-1, 0, 1):
        a, b = look(table, PFX_TR[k] + '|' + (BY_TR + d).astype(str))
        s, n = s + a, n + b
        a, b = look(table, PFX_TE[k] + '|' + (BY_TE + d).astype(str))
        st, nt = st + a, nt + b
    return (s - r) / (n - 1 + SHRINK[k]), st / (nt + SHRINK[k])


def f_decay(k, r, tau=3.0, max_gap=8):
    base_tr = PFX_TR[k] + '|' + BY_TR.astype(str) + '|'
    base_te = PFX_TE[k] + '|' + BY_TE.astype(str) + '|'
    table = sums(base_tr + YR_TR.astype(str), r)
    s, n, st, nt = np.zeros(n_train), np.zeros(n_train), np.zeros(len(TE)), np.zeros(len(TE))
    for d in range(-max_gap, max_gap + 1):
        w = float(np.exp(-abs(d) / tau))
        a, b = look(table, base_tr + (YR_TR + d).astype(str))
        s, n = s + w * a, n + w * b
        a, b = look(table, base_te + (YR_TE + d).astype(str))
        st, nt = st + w * a, nt + w * b
    return (s - r) / (n - 1 + SHRINK[k]), st / (nt + SHRINK[k])     # d = 0 时自己权重为 1，已扣除


def f_meet(r, lam=20.0):
    table = sums(MEET_TR, r)
    s, n = look(table, MEET_TR)
    st, nt = look(table, MEET_TE)
    return (s - r) / (n - 1 + lam), st / (nt + lam)


def s3_features(r):
    cols_tr, cols_te = [], []
    for builder in (f_exact, f_fuzzy, f_decay):
        for k in PREFIX:
            a, b = builder(k, r)
            cols_tr.append(a * VALID_TR)
            cols_te.append(b * VALID_TE)
    a, b = f_meet(r)
    cols_tr.append(a)
    cols_te.append(b)
    return np.column_stack(cols_tr), np.column_stack(cols_te)




def stage2(oof, test_pred):
    alphas = {lv: fit_alpha(LEVEL == lv, oof, MATE) for lv in ['exact', 'core']}
    p1 = apply_twins(oof, MATE, LEVEL, alphas)
    p1_cf = oof.copy()
    for h in (0, 1):
        a_h = {lv: fit_alpha((HALF != h) & (LEVEL == lv), oof, MATE) for lv in ['exact', 'core']}
        m = HALF == h
        p1_cf[m] = apply_twins(oof[m], MATE[m], LEVEL[m], a_h)
    r1 = y - p1
    F, FT = s3_features(r1)
    p2_cf = p1_cf.copy()
    for h in (0, 1):
        fit, m = HALF != h, HALF == h
        p2_cf[m] = p1_cf[m] + F[m] @ np.linalg.lstsq(F[fit], r1[fit], rcond=None)[0]
    beta = np.linalg.lstsq(F, r1, rcond=None)[0]
    test_final = apply_twins(test_pred, T_MATE, T_LEVEL, alphas) + FT @ beta
    return {'oof': rmse(y, oof), '+双胞胎': rmse(y, p1_cf), '+S3': rmse(y, p2_cf),
            'fold_final': fold_rmse(p2_cf), 'test_final': test_final}


net_rows, NET = [], {}
for name in [v for v in ['R3', FINAL4] if v is not None]:
    NET[name] = stage2(BLENDS[name]['oof'], BLENDS[name]['test'])
    net_rows.append({'variant': name, **{k: NET[name][k] for k in ['oof', '+双胞胎', '+S3']}})
    path = OUTPUT_DIR / f'submission_{name}_blend3_stage2.csv'
    pd.DataFrame({'id': test['id'], TARGET: np.clip(NET[name]['test_final'], 0, None)}).to_csv(path, index=False)
    print('写出', path.name)
net_table = pd.DataFrame(net_rows).set_index('variant')
display(net_table.round(3))
if FINAL4 is not None:
    delta = NET[FINAL4]['fold_final'] - NET['R3']['fold_final']
    print(f'{FINAL4} 相对 R3（都加平滑后）：平均 {delta.mean():+.3f}，{int((delta < 0).sum())}/5 折更好')
net_table.to_csv(OUTPUT_DIR / 'summary_net_after_stage2.csv')

写出 submission_R3_blend3_stage2.csv
写出 submission_AS_blend3_stage2.csv


,oof,+双胞胎,+S3
variant,,,
R3,76.717,76.493,76.361
AS,76.856,76.699,76.508


AS 相对 R3（都加平滑后）：平均 +0.147，0/5 折更好


## 12. 目标变量工程：异速生长 / IPF GL 作为目标（R3 特征，五折）

In [ ]:
for model in TARGET_MODELS:
    if (model, 'R3') in failed:
        continue
    for fold in range(1, N_FOLDS + 1):
        info, how = run_one(model, 'R3', fold)
        print(f'fold {fold}  {model:8s}  RMSE {info["rmse"]:.3f}  最佳轮数 {info["best_iteration"]}  ({how})')

base_models = SCREEN_MODELS
P_base = np.column_stack([load_oof(m, 'R3')[0] for m in base_models])


def nested_nnls(P):
    out = np.zeros(n_train)
    for k in range(1, N_FOLDS + 1):
        fit, va = FOLD_IDS != k, FOLD_IDS == k
        w, _ = nnls(P[fit], y[fit])
        out[va] = P[va] @ w
    return out


base_nested = nested_nnls(P_base)
rows = [{'model': m, 'oof': rmse(y, load_oof(m, 'R3')[0])} for m in base_models]
for model in TARGET_MODELS:
    loaded = load_oof(model, 'R3')
    if loaded is None:
        continue
    nested = nested_nnls(np.column_stack([P_base, loaded[0]]))
    delta = fold_rmse(nested) - fold_rmse(base_nested)
    rows.append({'model': model, 'oof': rmse(y, loaded[0]),
                 'corr_with_cb_rel': float(np.corrcoef(loaded[0], load_oof('cb_rel', 'R3')[0])[0, 1]),
                 'blend_delta_nested': rmse(y, nested) - rmse(y, base_nested), 'folds_better': int((delta < 0).sum())})
target_table = pd.DataFrame(rows).set_index('model')
print(f'R3 三模型融合（嵌套）：{rmse(y, base_nested):.3f}')
display(target_table.round(3))
target_table.to_csv(OUTPUT_DIR / 'summary_target_engineering.csv')
print('完成。请提交 submission_R3_blend3_stage2.csv 和 submission_<新变体>_blend3_stage2.csv 做配对比较；跑完告诉 Claude。')

fold 1  cb_allo   RMSE 77.468  最佳轮数 2924  (114s)
fold 2  cb_allo   RMSE 77.354  最佳轮数 3023  (118s)
fold 3  cb_allo   RMSE 77.748  最佳轮数 2341  (93s)
fold 4  cb_allo   RMSE 77.351  最佳轮数 2187  (87s)
fold 5  cb_allo   RMSE 76.711  最佳轮数 2751  (108s)
fold 1  cb_gl     RMSE 77.283  最佳轮数 2709  (107s)
fold 2  cb_gl     RMSE 77.202  最佳轮数 2660  (105s)
fold 3  cb_gl     RMSE 77.597  最佳轮数 2821  (114s)
fold 4  cb_gl     RMSE 77.220  最佳轮数 2298  (95s)
fold 5  cb_gl     RMSE 76.505  最佳轮数 2168  (91s)
R3 三模型融合（嵌套）：76.723


,oof,corr_with_cb_rel,blend_delta_nested,folds_better
model,,,,
lgb,77.709,NaN,NaN,NaN
xgb,78.458,NaN,NaN,NaN
cb_rel,77.160,NaN,NaN,NaN
cb_allo,77.327,0.997,-0.043,5.0
cb_gl,77.162,0.998,-0.076,5.0


完成。请提交 submission_R3_blend3_stage2.csv 和 submission_<新变体>_blend3_stage2.csv 做配对比较；跑完告诉 Claude。
